# Remaining Work Notebook: Robustness and Multi-Seed Study

This notebook is independent of `01_dataset_download.ipynb`. It focuses only on the remaining research tasks: secondary-dataset robustness evaluation and multi-seed comparison.

## Scope

1. Download a proposal-aligned secondary dataset into `data/raw/secondary/source`
2. Normalize and audit the secondary dataset
3. Evaluate the best trained checkpoint on the secondary dataset
4. Run a multi-seed study for the baseline CNN and TumorDetNet
5. Save CSVs, JSON summaries, and figures under `outputs/`

In [ ]:
from __future__ import annotations

import logging
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, clear_output, display
from tqdm import tqdm

tqdm.monitor_interval = 0


In [ ]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'pyproject.toml').exists() else current_dir.parent
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    raise RuntimeError(f'Could not resolve project root from {current_dir}')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CACHE_DIR = PROJECT_ROOT / 'outputs' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['BRAIN_TUMOR_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['XDG_CACHE_HOME'] = str(CACHE_DIR)
os.environ['TORCH_HOME'] = str(CACHE_DIR / 'torch')
os.environ['MPLCONFIGDIR'] = str(CACHE_DIR / 'matplotlib')

logger = logging.getLogger('remaining_work_notebook')
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s'))
logger.addHandler(handler)
logger.propagate = False

logger.info('Project root: %s', PROJECT_ROOT)

2026-03-27 16:25:21,694 | INFO | remaining_work_notebook | Project root: F:\brain tumor detection


In [ ]:
import importlib

from src.config.settings import load_settings
import src.experiments.multiseed as multiseed_module
import src.experiments.robustness as robustness_module

importlib.reload(multiseed_module)
importlib.reload(robustness_module)

run_multiseed_study = multiseed_module.run_multiseed_study
prepare_secondary_dataset = robustness_module.prepare_secondary_dataset
evaluate_on_secondary_dataset = robustness_module.evaluate_on_secondary_dataset

settings = load_settings(PROJECT_ROOT / 'src' / 'config' / 'default.yaml')
settings.data['paths']['root'] = str(PROJECT_ROOT)

SECONDARY_SOURCE_DIR = PROJECT_ROOT / 'data' / 'raw' / 'secondary' / 'source'
SECONDARY_DOWNLOAD_DIR = PROJECT_ROOT / 'data' / 'raw' / 'secondary' / 'downloads'
SECONDARY_NORMALIZED_DIR = PROJECT_ROOT / 'data' / 'raw' / 'secondary' / 'normalized'
SECONDARY_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
SECONDARY_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
SECONDARY_NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

MULTISEED_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed' / 'multiseed'
MULTISEED_METRICS_DIR = PROJECT_ROOT / 'outputs' / 'metrics' / 'multiseed'
MULTISEED_FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / 'multiseed'
MULTISEED_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MULTISEED_METRICS_DIR.mkdir(parents=True, exist_ok=True)
MULTISEED_FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Step 0: Download the Secondary Dataset

Chosen dataset: `sartajbhuvaji/brain-tumor-classification-mri` from Kaggle.

Why this dataset was chosen for the proposal:

- it is a public MRI dataset
- it is distinct from the primary dataset
- it uses the same four classes needed for cross-dataset validation: glioma, meningioma, pituitary, and no tumor
- it fits the proposal requirement for alternate public dataset testing under domain variation

If the dataset already exists in `data/raw/secondary/source/`, this cell will skip the download.

In [ ]:
SECONDARY_DATASET_SLUG = 'sartajbhuvaji/brain-tumor-classification-mri'

def secondary_source_exists() -> bool:
    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
    for path in SECONDARY_SOURCE_DIR.rglob('*'):
        if path.is_file() and path.suffix.lower() in valid_extensions:
            return True
    return False


def clear_directory_contents(directory: Path) -> None:
    directory.mkdir(parents=True, exist_ok=True)
    for child in directory.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()


def ensure_kaggle_available() -> None:
    try:
        import kaggle  # noqa: F401
        logger.info('Kaggle package already installed.')
    except ImportError:
        logger.info('Installing Kaggle package into the current environment.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle'], check=True)


def download_secondary_dataset() -> Path | None:
    if secondary_source_exists():
        logger.info('Secondary dataset already exists in %s. Skipping download.', SECONDARY_SOURCE_DIR)
        return None

    logger.info(
        'Using %s as the secondary dataset because it is a public alternate MRI dataset with the same four target classes.',
        SECONDARY_DATASET_SLUG,
    )

    archive_path = SECONDARY_DOWNLOAD_DIR / 'brain-tumor-classification-mri.zip'
    command = [
        sys.executable,
        '-m',
        'kaggle.cli',
        'datasets',
        'download',
        '-d',
        SECONDARY_DATASET_SLUG,
        '-p',
        str(SECONDARY_DOWNLOAD_DIR),
        '--force',
    ]

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
    )

    output_lines: list[str] = []
    progress = tqdm(total=100, desc='Downloading secondary dataset', unit='%')
    last_percent = 0

    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.strip()
        if not line:
            continue
        output_lines.append(line)
        logger.info('%s', line)
        match = re.search(r'(\d+)%', line)
        if match:
            percent = int(match.group(1))
            if percent > last_percent:
                progress.update(percent - last_percent)
                last_percent = percent

    return_code = process.wait()
    if return_code == 0 and last_percent < 100:
        progress.update(100 - last_percent)
    progress.close()

    if return_code != 0:
        raise RuntimeError('Secondary dataset download failed:\n' + '\n'.join(output_lines))

    if not archive_path.exists():
        zip_files = sorted(SECONDARY_DOWNLOAD_DIR.glob('*.zip'))
        if len(zip_files) != 1:
            raise FileNotFoundError(
                f'Expected one secondary dataset zip in {SECONDARY_DOWNLOAD_DIR}, found {len(zip_files)}'
            )
        archive_path = zip_files[0]

    clear_directory_contents(SECONDARY_SOURCE_DIR)
    logger.info('Extracting %s into %s', archive_path, SECONDARY_SOURCE_DIR)
    shutil.unpack_archive(str(archive_path), str(SECONDARY_SOURCE_DIR))
    return archive_path


ensure_kaggle_available()
secondary_archive = download_secondary_dataset()
secondary_archive

2026-03-27 16:25:25,919 | INFO | remaining_work_notebook | Kaggle package already installed.
2026-03-27 16:25:25,921 | INFO | remaining_work_notebook | Secondary dataset already exists in F:\brain tumor detection\data\raw\secondary\source. Skipping download.


## Step 1: Prepare the Secondary Dataset

Put the raw secondary dataset under `data/raw/secondary/source/`. This cell normalizes nested folder layouts into `data/raw/secondary/normalized/` and saves a summary.

In [ ]:
secondary_result = prepare_secondary_dataset(
    source_dir=SECONDARY_SOURCE_DIR,
    normalized_dir=SECONDARY_NORMALIZED_DIR,
    metrics_dir=PROJECT_ROOT / 'outputs' / 'metrics',
    figures_dir=PROJECT_ROOT / 'outputs' / 'figures',
    valid_extensions=tuple(ext.lower() for ext in settings.dataset['valid_extensions']),
)
logger.info('Secondary dataset summary: %s', secondary_result['summary'])
secondary_result

Normalizing secondary dataset: 100%|##########| 3264/3264 [00:04<00:00, 656.70it/s] 


2026-03-27 16:25:48,821 | INFO | remaining_work_notebook | Secondary dataset summary: {'num_images': 3160, 'num_classes': 4, 'classes': {'glioma': 926, 'meningioma': 937, 'notumor': 396, 'pituitary': 901}, 'image_width_range': [174, 1375], 'image_height_range': [167, 1446]}


{'summary': {'num_images': 3160,
  'num_classes': 4,
  'classes': {'glioma': 926,
   'meningioma': 937,
   'notumor': 396,
   'pituitary': 901},
  'image_width_range': [174, 1375],
  'image_height_range': [167, 1446]},
 'inventory_path': WindowsPath('F:/brain tumor detection/outputs/metrics/secondary_normalized_inventory.csv'),
 'summary_path': WindowsPath('F:/brain tumor detection/outputs/metrics/secondary_dataset_summary.json'),
 'class_counts_path': WindowsPath('F:/brain tumor detection/outputs/metrics/secondary_class_counts.csv'),
 'class_plot_path': WindowsPath('F:/brain tumor detection/outputs/figures/secondary_class_distribution.png'),
 'normalized_dir': WindowsPath('F:/brain tumor detection/data/raw/secondary/normalized')}

## Step 2: Robustness Evaluation on the Secondary Dataset

This evaluates the best `TumorDetNet` checkpoint on the normalized secondary dataset and saves a drift comparison against the primary test metrics.

In [ ]:
secondary_eval = evaluate_on_secondary_dataset(
    settings=settings,
    model_name='tumordetnet',
    checkpoint_path=PROJECT_ROOT / 'outputs' / 'models' / 'tumordetnet_best.pt',
    primary_results_path=PROJECT_ROOT / 'outputs' / 'metrics' / 'tumordetnet_results.json',
    secondary_dataset_dir=SECONDARY_NORMALIZED_DIR,
    output_metrics_path=PROJECT_ROOT / 'outputs' / 'metrics' / 'tumordetnet_secondary_robustness.json',
    output_drift_csv_path=PROJECT_ROOT / 'outputs' / 'metrics' / 'tumordetnet_secondary_drift.csv',
    output_confusion_path=PROJECT_ROOT / 'outputs' / 'figures' / 'tumordetnet_secondary_confusion_matrix.png',
)
secondary_eval['secondary_metrics']

F:\brain tumor detection\src\experiments\robustness.py:222: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=drift_plot_df, x="metric", y="delta_secondary_minus_primary", palette="flare")


{'accuracy': 0.8430379746835444,
 'precision': 0.8623266447112728,
 'recall': 0.8430379746835444,
 'f1_score': 0.8466389860491865,
 'confusion_matrix': [[711, 152, 54, 9],
  [13, 786, 100, 38],
  [6, 42, 344, 4],
  [3, 37, 38, 823]],
 'roc_auc': 0.9635046175721128,
 'class_metrics': {'glioma': {'precision': 0.9699863574351978,
   'recall': 0.7678185745140389,
   'f1_score': 0.8571428571428571},
  'meningioma': {'precision': 0.7728613569321534,
   'recall': 0.8388473852721452,
   'f1_score': 0.804503582395087},
  'notumor': {'precision': 0.6417910447761194,
   'recall': 0.8686868686868687,
   'f1_score': 0.7381974248927039},
  'pituitary': {'precision': 0.9416475972540046,
   'recall': 0.9134295227524972,
   'f1_score': 0.9273239436619718}},
 'loss': 0.4494984329147618,
 'accuracy_from_epoch': 0.8430379746835444}

## Step 3: Multi-Seed Study

This runs fresh training for each model across multiple seeds and saves run-level CSVs, summary CSVs, JSON, and boxplots.

In [ ]:
SEEDS = [42, 52, 62]
MODELS = ['baseline_cnn', 'tumordetnet']

run_bar = tqdm(total=len(SEEDS) * len(MODELS), desc='multi-seed runs', unit='run')
epoch_bar = tqdm(total=1, desc='epoch progress', unit='epoch', leave=False)
phase_bar = tqdm(total=1, desc='phase progress', unit='batch', leave=False)
current_run = {'model': None, 'seed': None}
completed_lines: list[str] = []

def multiseed_progress(event: dict[str, object]) -> None:
    event_name = str(event['event'])
    if event_name == 'run_start':
        current_run['model'] = event['model_name']
        current_run['seed'] = event['seed']
        epoch_bar.reset(total=int(settings.training['epochs']))
        epoch_bar.set_description(f"{event['model_name']} seed {event['seed']}")
        epoch_bar.refresh()
    elif event_name == 'epoch_start':
        target = int(event['epoch'])
        increment = target - epoch_bar.n
        if increment > 0:
            epoch_bar.update(increment)
    elif event_name == 'phase_start':
        phase_bar.reset(total=int(event['total_batches']))
        phase_bar.set_description(f"{current_run['model']} {event['phase']} e{event['epoch']}/{event['total_epochs']}")
        phase_bar.refresh()
    elif event_name == 'phase_progress':
        increment = int(event['batch']) - phase_bar.n
        if increment > 0:
            phase_bar.update(increment)
    elif event_name == 'phase_end':
        if phase_bar.total is not None and phase_bar.n < phase_bar.total:
            phase_bar.update(phase_bar.total - phase_bar.n)
    elif event_name == 'run_end':
        run_bar.update(1)
        completed_lines.append(
            f"Completed {event['model_name']} seed {event['seed']} | "
            f"val acc {event['validation_metrics']['accuracy']:.4f} | "
            f"test acc {event['test_metrics']['accuracy']:.4f}"
        )

try:
    multiseed_result = run_multiseed_study(
        settings=settings,
        primary_dataset_dir=PROJECT_ROOT / settings.paths['primary_dataset_dir'],
        processed_dir=MULTISEED_PROCESSED_DIR,
        models_dir=PROJECT_ROOT / 'outputs' / 'models',
        metrics_dir=MULTISEED_METRICS_DIR,
        figures_dir=MULTISEED_FIGURES_DIR,
        seeds=SEEDS,
        model_names=MODELS,
        progress_callback=multiseed_progress,
    )
finally:
    run_bar.close()
    epoch_bar.close()
    phase_bar.close()

clear_output(wait=True)
display(Markdown('### Multi-seed Runs Completed'))
for line in completed_lines:
    print(line)

summary_df = multiseed_result['summary_df'].copy()
summary_df = summary_df.round(4)

validation_view = summary_df[summary_df['split'] == 'validation'][[
    'model', 'accuracy_mean', 'accuracy_std', 'f1_score_mean', 'f1_score_std', 'roc_auc_mean', 'roc_auc_std', 'loss_mean', 'loss_std'
]].reset_index(drop=True)

test_view = summary_df[summary_df['split'] == 'test'][[
    'model', 'accuracy_mean', 'accuracy_std', 'f1_score_mean', 'f1_score_std', 'roc_auc_mean', 'roc_auc_std', 'loss_mean', 'loss_std'
]].reset_index(drop=True)

display(Markdown('### Validation Summary'))
display(validation_view)
display(Markdown('### Test Summary'))
display(test_view)


multi-seed runs:   0%|          | 0/6 [00:00<?, ?run/s]

epoch progress:   0%|          | 0/1 [00:00<?, ?epoch/s]

phase progress:   0%|          | 0/1 [00:00<?, ?batch/s]

Completed baseline_cnn seed 42 | val acc 0.8139 | test acc 0.8333
Completed tumordetnet seed 42 | val acc 0.8694 | test acc 0.8750
Completed baseline_cnn seed 52 | val acc 0.8046 | test acc 0.8037
Completed tumordetnet seed 52 | val acc 0.9037 | test acc 0.9148


Completed baseline_cnn seed 62 | val acc 0.8000 | test acc 0.8102
Completed tumordetnet seed 62 | val acc 0.8954 | test acc 0.8852


,model,split,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_score_mean,f1_score_std,roc_auc_mean,roc_auc_std,loss_mean,loss_std
0,baseline_cnn,test,0.815741,0.015576,0.825045,0.019621,0.815741,0.015576,0.814579,0.017722,0.953965,0.004684,0.489591,0.030934
1,baseline_cnn,validation,0.806173,0.007072,0.816123,0.009006,0.806173,0.007072,0.805056,0.008352,0.952097,0.007313,0.495004,0.031769
2,tumordetnet,test,0.891667,0.020684,0.893903,0.019342,0.891667,0.020684,0.890657,0.021310,0.983330,0.003594,0.302356,0.043325
3,tumordetnet,validation,0.889506,0.017867,0.892632,0.015363,0.889506,0.017867,0.888359,0.019167,0.982582,0.001747,0.313953,0.039146


## Outputs from This Notebook

- Secondary dataset inventory and summary in `outputs/metrics/`
- Secondary robustness metrics and drift CSV in `outputs/metrics/`
- Secondary class distribution and confusion matrix figures in `outputs/figures/`
- Multi-seed run CSVs and summary CSV in `outputs/metrics/multiseed/`
- Multi-seed comparison plots in `outputs/figures/multiseed/`